Transform Customer Data  -- Lesson 47

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct AS
SELECT DISTINCT * 
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;


In [0]:
%sql

SELECT 
  customer_id, 
  MAX(created_timestamp) AS max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id;

In [0]:
%sql
WITH cte_max AS  -- deduplicate data
(
SELECT 
  customer_id, 
  MAX(created_timestamp) AS max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT t.* 
FROM v_customers_distinct t
JOIN cte_max  m 
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp;  -- only select the ones that are the most recent timestamp


CAST Column timestamp

In [0]:
%sql
WITH cte_max AS  -- deduplicate data
(
SELECT 
  customer_id, 
  MAX(created_timestamp) AS max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT CAST(t.created_timestamp as TIMESTAMP) AS created_timestamp,
      t.customer_id,
      t.customer_name,
      CAST(t.date_of_birth as DATE) AS date_of_birth,
      CAST(t.member_since as DATE) AS member_since,
      t.telephone,
      t.email
FROM v_customers_distinct t
JOIN cte_max  m 
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp;  -- only select the ones that are the most recent timestamp


### 5. Write Data as Delta Table

In [0]:
%sql

CREATE TABLE gizmobox.silver.customers AS
WITH cte_max AS  -- deduplicate data
(
SELECT 
  customer_id, 
  MAX(created_timestamp) AS max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT CAST(t.created_timestamp as TIMESTAMP) AS created_timestamp,
      t.customer_id,
      t.customer_name,
      CAST(t.date_of_birth as DATE) AS date_of_birth,
      t.email,
      CAST(t.member_since as DATE) AS member_since,
      t.telephone
FROM v_customers_distinct t
JOIN cte_max  m 
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp;  -- only select the ones that are the most recent timestamp

In [0]:
%sql


SELECT * FROM gizmobox.silver.customers;

In [0]:
%sql
DESCRIBE EXTENDED gizmobox.silver.customers;